In [ ]:
%reload_ext autoreload
%autoreload 2
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from vivarium import InteractiveContext

In [35]:
%cd /home/abie/vivarium_nih_moud/tests/

/home/abie/vivarium_nih_moud/tests


/home/abie/miniforge3/envs/vivarium_nih_moud_simulation/lib/python3.11/site-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [36]:
!cat ../src/vivarium_nih_moud/model_specifications/model_spec.yaml

components:
    vivarium_public_health.population:
        - BasePopulation()
        - Mortality()
    vivarium_nih_moud.components:
        - conditions.moud_model()
        - locations.quarters_model()
        - locations.quarters_risk()
        - locations.SimpleRiskEffect('quarters_risk',
                'oud_consistent_to_on_treatment_for_oud_consistent.transition_rate')
    vivarium_public_health.results:
        - ResultsStratifier()
        - DisabilityObserver()
        - DiseaseObserver('oud_consistent')
        - DiseaseObserver('quarters')
        - MortalityObserver()

configuration:
    input_data:
        input_draw_number: 0
        artifact_path: /home/abie/vivarium_nih_moud/washington.hdf
        # artifact_path: /mnt/team/simulation_science/pub/training/abie/artifacts/washington.hdf

    interpolation:
        order: 0
        extrapolate: True
    randomness:
        map_size: 1_000_000
        key_columns: ['entrance_time', 'age']
        random_seed: 4344
       

In [37]:
sim = InteractiveContext('../src/vivarium_nih_moud/model_specifications/model_spec.yaml')

2025-06-08 04:21:45.112 | INFO     | simulation_3-artifact_manager:77 - Running simulation from artifact located at /home/abie/vivarium_nih_moud/washington.hdf.
2025-06-08 04:21:45.115 | INFO     | simulation_3-artifact_manager:78 - Artifact base filter terms are ['draw == 0'].
2025-06-08 04:21:45.115 | INFO     | simulation_3-artifact_manager:79 - Artifact additional filter terms are None.


/home/abie/miniforge3/envs/vivarium_nih_moud_simulation/lib/python3.11/site-packages/vivarium_public_health/population/data_transformations.py:57: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the previous behavior, use

	>>> .groupby(..., group_keys=False)

To adopt the future behavior and silence this warning, use 

	>>> .groupby(..., group_keys=True)
  .apply(lambda sub_pop: sub_pop[["value"]] / sub_pop["value"].sum())
/home/abie/miniforge3/envs/vivarium_nih_moud_simulation/lib/python3.11/site-packages/vivarium_public_health/population/data_transformations.py:64: FutureWarning: Not prepending group keys to the result index of transform-like apply. In the future, the group keys will be included in the index, regardless of whether the applied function returns a like-indexed object.
To preserve the pre

2025-06-08 04:21:47.863 | WARNING  | simulation_3-resource_manager:176 - Resource stream.oud_consistent_initial_states is not produced by any component but is needed to compute (column.oud_consistent).
2025-06-08 04:21:47.865 | WARNING  | simulation_3-resource_manager:176 - Resource stream.quarters_initial_states is not produced by any component but is needed to compute (column.quarters).


/home/abie/miniforge3/envs/vivarium_nih_moud_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:114: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(
/home/abie/miniforge3/envs/vivarium_nih_moud_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:114: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grouper equal to a list of length 1. Don't supply a list with a single grouper to avoid this warning.
  sub_tables = list(
/home/abie/miniforge3/envs/vivarium_nih_moud_simulation/lib/python3.11/site-packages/vivarium/framework/lookup/interpolation.py:114: FutureWarning: In a future version of pandas, a length 1 tuple will be returned when iterating over a groupby with a grou

In [ ]:
pop0 = sim.get_population()
pop0

In [ ]:
pop0.oud_consistent.value_counts(normalize=True)

In [ ]:
pop0.oud_consistent.value_counts()

In [ ]:
pop0.quarters.value_counts()

In [ ]:
sim.take_steps(10)

In [ ]:
pop1 = sim.get_population()

In [ ]:
pop1.oud_consistent.value_counts()

In [ ]:
pop1.quarters.value_counts()

In [ ]:
pd.crosstab(pop0.oud_consistent, pop0.quarters, margins=True)

In [ ]:
np.round(100*pd.crosstab(pop1.oud_consistent, pop1.quarters, margins=True, normalize='index'), 1)

In [ ]:
np.round(100*pd.crosstab(pop1.quarters, pop1.oud_consistent, margins=True, normalize='index'), 1)

# Goal: induce corelation between quarters and oud state

Approach: follow pattern similar to the Risk and RiskEffect models.

In [ ]:
sim.step()

In [ ]:
pop2 = sim.get_population()

In [ ]:
t = pd.concat([pop0, pop1, pop2], axis=1).filter(like='quarters')
t[t.iloc[:, 2] != 'housed']

In [ ]:
[x for x in sim.list_values() if 'expo' in x]

In [ ]:
%cd /home/abie/vivarium_results/washington/2025_06_05_14_05_18/results
!ls -halt

In [ ]:
# pd.read_parquet('deaths.parquet').sort_values('value', ascending=False)

In [ ]:
# pd.read_parquet('ylls.parquet').sort_values('value', ascending=False)

In [ ]:
df_t = pd.read_parquet('transition_count_oud_consistent.parquet').sort_values('value')
df_t

In [ ]:
df_t[df_t.sub_entity == 'oud_consistent_to_on_treatment_for_oud_consistent'].groupby('quarters').sum()

In [ ]:
# pd.read_parquet('transition_count_quarters.parquet').sort_values('value')

In [ ]:
df_pt = pd.read_parquet('person_time_quarters.parquet').sort_values('value')
df_pt

In [ ]:
df_pt[df_pt.oud_consistent == 'oud_consistent'].groupby('sub_entity').sum()

In [ ]:
(
    df_t[df_t.sub_entity == 'oud_consistent_to_on_treatment_for_oud_consistent'].groupby('quarters').sum()
    /
    df_pt[df_pt.oud_consistent == 'oud_consistent'].groupby('sub_entity').sum()
)

In [20]:
df = pd.read_parquet('person_time_oud_consistent.parquet').sort_values('value')

In [32]:
def plot_prevalence(df):
    assert np.all(df.measure == 'person_time')
    denom = df.groupby(['sex', 'age_group', 'current_year']).value.sum()
    numer = df[df.sub_entity != 'susceptible_to_oud_consistent'].groupby(['sex', 'age_group', 'current_year']).value.sum()
    prevalence = numer / denom
    t = prevalence.unstack()
    return np.round(t[t['2019'] > 0]*100, 1)
plot_prevalence(df)

current_year      2019  2020
sex    age_group            
Female 15_to_19    0.9   NaN
       20_to_24    2.4   NaN
       25_to_29    5.4   NaN
       30_to_34    4.7   NaN
       35_to_39    4.6   NaN
       40_to_44    3.4   NaN
       45_to_49    3.2   NaN
       50_to_54    2.7   NaN
Male   15_to_19    0.9   NaN
       20_to_24    2.4   NaN
       25_to_29    5.5   NaN
       30_to_34    4.5   NaN
       35_to_39    4.0   NaN
       40_to_44    2.3   NaN
       45_to_49    1.6   NaN
       50_to_54    1.3   NaN

In [38]:
from vivarium import Artifact
artifact_path = sim.configuration.input_data.artifact_path
art = Artifact(artifact_path)


In [42]:
cause = 'oud_consistent'
df_art = art.load(f"cause.{cause}.prevalence")


In [49]:
df_art[f'draw_{sim.configuration.input_data.input_draw_number}']

sex     age_start  age_end  year_start  year_end
Male    0          5        2020        2021        2.062810e-09
        5          10       2020        2021        1.148598e-08
        10         15       2020        2021        1.234720e-08
        15         20       2020        2021        8.994984e-03
        20         25       2020        2021        2.752567e-02
        25         30       2020        2021        5.629641e-02
        30         35       2020        2021        4.349930e-02
        35         40       2020        2021        3.974496e-02
        40         45       2020        2021        2.554370e-02
        45         50       2020        2021        1.944922e-02
        50         55       2020        2021        1.480591e-02
        55         60       2020        2021        1.093584e-02
        60         65       2020        2021        5.426230e-03
        65         70       2020        2021        2.754602e-03
        70         75       2020        2